In [1]:
import time
import numpy as np
import serial
import csv
import matplotlib.pyplot as plt

def write_value(ser, command_root, value, print_setup=False):
    command = command_root + bytes(" {value}\n".format(value=value), "utf-8")
    #print(command)
    ser.write(command)
    ser.read(len(command)-1)
    if print_setup==True:
        ser.write(command_root+b"?\n")
        print(ser.readline())

def get_voltage(arduino_ser):
    arduino_ser.write(b"READ\n")
    line = arduino_ser.readline().decode(errors="ignore").strip()
    try:
        voltage = float(line)
    except:
        return None
    #print(f"Received from Arduino: {line}")
    return voltage

In [2]:
#check which serial ports are available
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()

for port in ports:
    print(f"Device: {port.device}")
    print(f"Name: {port.name}")
    print(f"Description: {port.description}")
    print(f"HWID: {port.hwid}")
    print("-" * 40)

Device: /dev/ttyS3
Name: ttyS3
Description: n/a
HWID: n/a
----------------------------------------
Device: /dev/ttyS2
Name: ttyS2
Description: n/a
HWID: n/a
----------------------------------------
Device: /dev/ttyS1
Name: ttyS1
Description: n/a
HWID: n/a
----------------------------------------
Device: /dev/ttyS0
Name: ttyS0
Description: n/a
HWID: n/a
----------------------------------------
Device: /dev/ttyUSB0
Name: ttyUSB0
Description: FT232R USB UART - FT232R USB UART
HWID: USB VID:PID=0403:6001 SER=A8006Avp LOCATION=1-6.2.4
----------------------------------------
Device: /dev/ttyACM1
Name: ttyACM1
Description: ttyACM1
HWID: USB VID:PID=2341:0042 SER=7413234353035131A0B0 LOCATION=1-6.1:1.0
----------------------------------------
Device: /dev/ttyACM0
Name: ttyACM0
Description: ttyACM0
HWID: USB VID:PID=2341:0043 SER=85731303533351C0C231 LOCATION=1-2:1.0
----------------------------------------


In [7]:
laser_port = "/dev/ttyUSB0"
arduino_port = "/dev/ttyACM1"
laser_ser = serial.Serial(laser_port, 38400, timeout=1, stopbits=1, parity="N")
laser_ser.close()
arduino_ser = serial.Serial(arduino_port, 9600, timeout=1)
try:
    arduino_ser.close()
except:
    pass


In [8]:
time.sleep(1)
laser_ser = serial.Serial(laser_port, 38400, timeout=1, stopbits=1, parity="N")
print(laser_ser.name)

#check if we are connected to the right device
laser_ser.write(b'*IDN?\n')
ser_no = laser_ser.readline()
if ser_no != b'*IDN?Arroyo 4205-DR 110802133 v2.01a 211\n':
    print("Wrong device")
    laser_ser.close()

#set the limits for the laser
write_value(laser_ser, b'LASer:LIMit:LDV', 3.2, print_setup=True )
write_value(laser_ser, b'LASer:LIMit:LDI', 100, print_setup=True)
write_value(laser_ser, b"LASer:STEP", 1, print_setup=True)
print("Laser setup complete")

#set the initial current to 72 mA, after that wait for 10s for the laser's temperature to stabilize
I_init_mA = 71
write_value(laser_ser, b'LASer:LDI', I_init_mA)
time.sleep(5)

I_final_mA=74
steps_range = int((I_final_mA- I_init_mA)/0.01)
print("Laser initialized")



#open the serial connection to the arduino and reset the input buffer
#wait for 2 seconds for the connection to be established
#throw away the first reading

arduino_ser = serial.Serial(arduino_port, 9600, timeout=1)
time.sleep(2)
arduino_ser.reset_input_buffer()
get_voltage(arduino_ser)
print("Arduino initialized")


#data acquisition loop: increase the current by 0.01 mA every second for 100 seconds, and record the voltage at each step
with open("F4_data.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["current_mA", "voltage_V"])
    #increase the current by 0.01 mA every second for 100 seconds
    for i in range(400):    
        laser_ser.write(b"LASer:INC 1\n")
        time.sleep(0.8)
        current_mA = I_init_mA + (i+1) * 0.01
        voltage = get_voltage(arduino_ser)
        #print(f"Current: {current_mA:.2f} mA, Voltage: {voltage} V")
        writer.writerow([f"{current_mA:.2f}", f"{voltage}"])
write_value(laser_ser, b'LASer:LDI', I_init_mA)
print("Data acquisition Complete")

/dev/ttyUSB0
b'LASer:LIMit:LDV?3.2\n'
b'LASer:LIMit:LDI?100\n'
b'LASer:STEP?1\n'
Laser setup complete
Laser initialized
Arduino initialized
Data acquisition Complete
